# Flux-averaged cross sections versus W

SWAN notebook for the Upsilon early-science note. Keep the whole package together, check the shared profile, restart the kernel after updating the modules, then **Run All**.

Reads the selected reoptimization run's saved response-corrected bin-average cross sections, central toy sampling quantiles and full covariance at the selected run’s saved luminosity. It does not rescale yields or remake error bars. These bands are sampling quantiles, not confidence limits. A `NOT_VALIDATED` case requires explicit diagnostic permission and remains a diagnostic; preflight and the exported CSV/JSON retain its final status and failed checks.

The legend contains RECO and ZEUS. The Υ(1S) points and intervals are drawn above ZEUS. MC curves and legend entries are hidden in all three panels; their values remain in the numerical audit. The published ZEUS values are unchanged. W bin midpoints are display coordinates; values are photon-flux-weighted bin averages, with no additional division by bin width. The actual state-specific Q²/ID settings remain in the audit.

Set `display_rebin.1S.w` to `"keep_outer_bins"` or an explicit list of saved edges in the shared profile to merge W display bins. Set it to `false` to retain the saved bins. This merges the original estimator, not a new three-bin response fit or optimization. Cross-section weights and the full covariance are propagated. Asymmetric sampling limits come from sums of joint Poisson toy fits replayed with the original seed and checked against the saved summaries. Marginal interval endpoints are never added. The first use can take a few minutes; results are cached in `OUTPUT_DIR/rebin_cache`. No ROOT files or full framework rerun are needed. The source run's actual validation status is preserved; display merging cannot promote it.


**Repository profile:** copy `configs/plots_9x275.example.json` to `configs/plots.local.json` and edit that one file. Paths, runs, luminosity, selections, optional `display_rebin` and diagnostic permission are shared by all notebooks. See `docs/plotting.md`. Every execution creates a fresh output directory. The current figure layouts require all three states; the 1S-only 9×130 support run is reviewed through its status/audit outputs.


**Separate proton-tagged category.** The input reader requires the saved `jul26-reco-topology-v1` selection contract (topology 1/2 and finite positive tagged-proton momentum), applied to reconstruction only. Truth denominators and generated normalization remain unchanged. Run this notebook in a fresh kernel from this folder; do not mix modules or results with the original no-tag package.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, FileLink
from settings import load_profile, make_reoptimization_config, new_output_dir, apply_display_rebin, file_link_path
from upsilon_esr import preflight

# No ROOT or LaTeX installation is needed. If a Python dependency is missing,
# uncomment the next line, run it once, and restart the notebook kernel.
# %pip install -r requirements.txt


## Configuration

Notebooks 02/03 select one explicit reoptimization run for all states. Notebook 01 uses the separate tagged full-framework runs. This notebook requires a new tagged reoptimization run; it will stop if that run is absent.


In [ ]:
# All paths, run IDs, selection metadata and diagnostic permission come from one profile.
# Default: ../configs/plots.local.json. Alternatively set UPSILON_PLOT_PROFILE
# before starting the SWAN kernel, or give load_profile("/path/to/profile.json").
PROFILE = load_profile()
OUTPUT_DIR = new_output_dir(PROFILE, "w")
cfg = make_reoptimization_config(profile=PROFILE, output_dir=OUTPUT_DIR)
apply_display_rebin(cfg, PROFILE, "w")
print("Profile:", PROFILE["profile_path"])
print("Output directory:", OUTPUT_DIR)


## Validate inputs

Check the saved luminosity, beam/selection metadata, bin edges, conversions, full covariance and final case status. Missing or mismatched inputs raise an error; there is no fallback to old results. Nonvalidated but complete inference requires `allow_diagnostic_results: true` in the shared profile.


In [ ]:
# Optional display merges were applied once from PROFILE["display_rebin"].
# A false/absent setting keeps the saved bins; no later cell overrides it.
print("Display merges:", cfg.rebin_edges)


In [ ]:
display(preflight(cfg, section="w"))

## Publication typography

The shared style is tuned for the single-column figures in the note. Source sizes are 16 pt for axis titles, 14.5 pt for ticks, 14 pt for legends, 13 pt for annotations, and 12.5–13.5 pt inside the slope inset. These become approximately 5.7–7.4 pt at the paper column width. The sizes are centralized in `upsilon_esr/common.py` as `FONT_SIZES`. Replacing the plotting modules requires a kernel restart before rerunning.


## Render the figure

In [ ]:
from upsilon_esr.w_cross_section import plot_w_cross_section

fig, audit = plot_w_cross_section(cfg)
plt.show()


## Numerical check

In [ ]:
points = pd.read_csv(OUTPUT_DIR / "upsilon_es_w_cross_section_points.csv")
display(points.loc[points["displayed"], ["state", "W_low", "W_high", "sigma_gamma_p_pb",
    "expected_reconstructed_count", "interval_low_pb", "interval_high_pb", "overall_status"]])
for state, item in audit["states"].items():
    print(state, "conversion check:", item["normalization_check"]["status"],
          "overall:", item["sample_metadata"]["overall_status"],
          "omitted bins:", item["omitted_row_indices"])


## Saved outputs

PDF is vector output for the paper; PNG is a quick preview. CSV/JSON companions record the displayed numbers and assumptions. Copy the three PDFs into the note’s `figures/proton_tagged/` directory.

In [ ]:
print("Output directory:", OUTPUT_DIR.resolve())
for suffix in ("pdf", "png"):
    display(FileLink(str(OUTPUT_DIR / ("upsilon_es_w_cross_section." + suffix))))
display(FileLink(str(OUTPUT_DIR / "upsilon_es_w_cross_section_audit.json")))